In [28]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the base model
model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set device based on availability of GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
    trust_remote_code=True
)

# Load the adapter
model_peft = PeftModel.from_pretrained(model, "azam25/TinyLlama_instruct_generation")

Some parameters are on the meta device because they were offloaded to the cpu.


In [29]:
def generate_response(message, model, max_tokens=50):

    # Create prompt and encode input
    input = [
        {"role": "user", "content": message},
    ]
    prompt = tokenizer.apply_chat_template(input, tokenize=False) 
    encoded_input = tokenizer(prompt, return_tensors="pt", add_special_tokens=True) 

    # Ensure the input tensors are on the correct device (same as model)
    model_inputs = {key: value.to(device) for key, value in encoded_input.items()}  # Move inputs to the same device

    # Generate response with a limit on the new tokens only (response tokens)
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    # Decode output
    decoded_output = tokenizer.batch_decode(generated_ids) 

    # Extract assistant's response
    assistant_response = decoded_output[0].split("<|assistant|>")[1].split("</s>")[0].strip()

    return assistant_response


In [30]:
message = "Hello, who are you?"

# Generate and print response
response = generate_response(message, model)
print(response)

I am a computer program.


In [31]:
message = "Cool, how are you?"

# Generate and print response
response = generate_response(message, model)
print(response)

I'm doing well, how are you?


Some parameters are on the meta device because they were offloaded to the cpu.


Message utilisateur : Hello, who are you?
Réponse du chatbot : I'm not available for human interaction and am just doing this prompt as an automated response. However, here goes.... : Are you being watched by someone while I log out from my account? Oh dear that


In [33]:
from transformers import pipeline
generator = pipeline("text-generation", model="gpt2")
generator("how to be cool?", max_length=20)

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': "how to be cool? We've asked a couple of them; most of them declined an explanation."}]

In [34]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

prompt = "GPT2 is a model developed by OpenAI."

input_ids = tokenizer(prompt, return_tensors="pt").input_ids

gen_tokens = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.9,
    max_length=100,
)
gen_text = tokenizer.batch_decode(gen_tokens)[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [41]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Charger le tokenizer et le modèle de génération
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Texte d'entrée
text = "Hello, who are you?"

# Encoder l'entrée
encoded_input = tokenizer(text, return_tensors='pt')

# Générer la réponse
output = model.generate(
    encoded_input['input_ids'],  # L'ID d'entrée pour la génération
    max_length=50,  # Longueur maximale de la réponse générée
    num_return_sequences=1,  # Nombre de réponses à générer
    do_sample=True,  # Utilisation de l'échantillonnage pour la diversité
    temperature=0.7  # Paramètre de diversité dans la génération
)

# Décoder la réponse générée
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

# Afficher la réponse générée
print("Réponse générée :")
print(generated_text)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Réponse générée :
Hello, who are you?

Safari

I am, my name is Safari. I am a new member of the squad.

Safari

Well, first of all, I am a new member of
